<a href="https://colab.research.google.com/github/NeuromatchAcademy/course-content/blob/main/tutorials/W1D3_GeneralizedLinearModels/student/W1D3_Intro.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a> &nbsp; <a href="https://kaggle.com/kernels/welcome?src=https://raw.githubusercontent.com/NeuromatchAcademy/course-content/main/tutorials/W1D3_GeneralizedLinearModels/student/W1D3_Intro.ipynb" target="_parent"><img src="https://kaggle.com/static/images/open-in-kaggle.svg" alt="Open in Kaggle"/></a>

# はじめに

## 概要

皆さん、こんにちは。今日は『GLM』の日、Neuromatchのハイライト（本当に！）へようこそ！

今回は一般化線形モデル、いわゆるGLMについて話します。GLMの素晴らしいところは、いくつかの有用な機械学習アルゴリズムを統一するシンプルでエレガントな枠組みを提供し、神経科学における幅広いデータ解析の問題に取り組むことができる点です。機械学習アルゴリズムのスイスアーミーナイフのような存在だと考えてください。

特に、線形回帰、ロジスティック回帰、ポアソン回帰など、皆さんが知っているかもしれないいくつかのアルゴリズムは、すべてGLMの特別なケースに過ぎません。これらは、エンコーディング（外部の共変量から神経活動を予測する）やデコーディング（神経活動から行動や意図を予測する）を研究する際に非常に役立ちます。

さらに、GLMは機械学習の一般的な概念を教えるのにも最適です。コスト関数の最適化（コスト関数が凸であることの意味とその有用性）、過学習とは何か、それを見分ける方法（クロスバリデーション！）、過学習を防ぐ方法（正則化！）、どのような種類の正則化があるか（L1、L2）、そしてこれがベイズ推定とどうつながるかなどです。

最後に――スイスアーミーナイフのように――GLMには限界もあり、時にはより強力な手法が必要になることもあります。しかし、そうした場合でもGLMは、より複雑なアルゴリズム（例えばディープネット）と比較するためのベースラインを提供します。また、GLMはより複雑なアルゴリズムを構築するための構成要素としても使えます。

本日の構成は以下の通りです。
「イントロ」では、Christina Savin博士が神経科学における代表的な解析タスクの概要を説明し、それらが共通して持つ特徴とGLMでどのように取り組めるかを示します。彼女は一つのタスク（受容野フィッティング）を例に取り、最適化、過学習、正則化といった概念を解説します。

チュートリアル1の前半では、Anqi Qu博士（現在ジョージア工科大学の助教授）が受容野推定の例をさらに掘り下げ、このタスクの数学的詳細を説明します。スパイクトリガー平均（優れたがしばしば経験的に動機づけられる手法）がGLMの特別なケース、すなわち古典的な線形回帰と密接に関連していることを示します。チュートリアル2の後半では、ポアソンGLMがスパイクデータのエンコーディングモデルに非常に強力な方法であることを示します。

チュートリアル3では、彼女がロジスティック回帰（私のお気に入りの分類アルゴリズム）に移り、デコーディングにどう使えるかを示します。また、正則化と異なる正則化手法の違いについても説明します。
「アウトロ」では、Memming Park博士がまず自身の猫について話し、その後GLMの神経科学への応用についてより詳細に語ります。一般的な概要の後、単一ニューロンや神経集団のダイナミクスのモデリング、意思決定タスクにおけるニューロンの応答のモデリング、そしてニューロンから行動選択をデコードする方法について説明します。最後に――重要なことですが――GLMのパラメータを因果的に解釈する危険性など、GLMの解釈における潜在的な落とし穴についても触れます。最後にGLMの一般化の可能性についても触れます。

全体のコースとの関連では、「モデルフィッティング」の日が最も近く、そこで基本的なアイデアのいくつかがすでに紹介されます。GLMの日を終えた後は、他の日やタスクでもGLMが使われているのに気づくでしょう（必ずしも「GLM」と呼ばれていない場合もありますが）、例えば行動データのデコーディングやダイナミクスのモデルとしての構成要素としてなどです。

それでは、今日一日を楽しんでください！

## 前提知識

本日のチュートリアルでは、行列ベクトル積（[W0D3 T2](https://compneuro.neuromatch.io/tutorials/W0D3_LinearAlgebra/student/W0D3_Tutorial2.html)）、確率分布（[W0D5 T1](https://compneuro.neuromatch.io/tutorials/W0D5_Statistics/student/W0D5_Tutorial1.html)）、および最尤推定（[W0D5 T2](https://compneuro.neuromatch.io/tutorials/W0D3_LinearAlgebra/student/W0D3_Tutorial2.html) と [W1D3](https://compneuro.neuromatch.io/tutorials/W1D3_ModelFitting/chapter_title.html)）を使用します。また、刺激と応答のペアリング（[Neuro Video Series Stimulus Representation](https://compneuro.neuromatch.io/tutorials/W0D0_NeuroVideoSeries/student/W0D0_Tutorial10.html)）や動物の意思決定実験（[Neuro Video Series Behavioural Readouts](https://compneuro.neuromatch.io/tutorials/W0D0_NeuroVideoSeries/student/W0D0_Tutorial3.html)）も扱います。新しいPythonライブラリである[sklearn](https://scikit-learn.org/stable/)を使用します。

In [ ]:
# @title Install and import feedback gadget

!pip3 install vibecheck datatops --quiet

from vibecheck import DatatopsContentReviewContainer
def content_review(notebook_section: str):
    return DatatopsContentReviewContainer(
        "",  # No text prompt
        notebook_section,
        {
            "url": "https://pmyvdlilci.execute-api.us-east-1.amazonaws.com/klab",
            "name": "neuromatch_cn",
            "user_key": "y1x3mpx5",
        },
    ).render()


feedback_prefix = "W1D3_Intro"

## ビデオ

In [ ]:
# @markdown
from ipywidgets import widgets
from IPython.display import YouTubeVideo
from IPython.display import IFrame
from IPython.display import display


class PlayVideo(IFrame):
  def __init__(self, id, source, page=1, width=400, height=300, **kwargs):
    self.id = id
    if source == 'Bilibili':
      src = f'https://player.bilibili.com/player.html?bvid={id}&page={page}'
    elif source == 'Osf':
      src = f'https://mfr.ca-1.osf.io/render?url=https://osf.io/download/{id}/?direct%26mode=render'
    super(PlayVideo, self).__init__(src, width, height, **kwargs)


def display_videos(video_ids, W=400, H=300, fs=1):
  tab_contents = []
  for i, video_id in enumerate(video_ids):
    out = widgets.Output()
    with out:
      if video_ids[i][0] == 'Youtube':
        video = YouTubeVideo(id=video_ids[i][1], width=W,
                             height=H, fs=fs, rel=0)
        print(f'Video available at https://youtube.com/watch?v={video.id}')
      else:
        video = PlayVideo(id=video_ids[i][1], source=video_ids[i][0], width=W,
                          height=H, fs=fs, autoplay=False)
        if video_ids[i][0] == 'Bilibili':
          print(f'Video available at https://www.bilibili.com/video/{video.id}')
        elif video_ids[i][0] == 'Osf':
          print(f'Video available at https://osf.io/{video.id}')
      display(video)
    tab_contents.append(out)
  return tab_contents


video_ids = [('Youtube', 'm1w7oywzwpA'), ('Bilibili', 'BV1BK411H7ie')]
tab_contents = display_videos(video_ids, W=854, H=480)
tabs = widgets.Tab()
tabs.children = tab_contents
for i in range(len(tab_contents)):
  tabs.set_title(i, video_ids[i][0])
display(tabs)

## スライド

In [ ]:
# @markdown
from IPython.display import IFrame
link_id = "qxfz9"
print(f"If you want to download the slides: https://osf.io/download/{link_id}/")
IFrame(src=f"https://mfr.ca-1.osf.io/render?url=https://osf.io/{link_id}/?direct%26mode=render%26action=download%26mode=render", width=854, height=480)

In [ ]:
# @title Submit your feedback
content_review(f"{feedback_prefix}_Day_Intro")